# Google Colab Version: [Open this notebook in Google Colab](https://colab.research.google.com/github/parawaveio/parawave/blob/main/examples/03_advanced_synthetic_pipeline.ipynb)

# Advanced: Diverse Synthetic Data with SharedState

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/parawaveio/parawave/blob/main/examples/03_advanced_synthetic_pipeline.ipynb)

Generate synthetic QA pairs and compare diversity with and without SharedState.
When SharedState feeds previous results back into the prompt, the LLM produces more diverse outputs.

In [1]:
!pip install parawave openai jinja2


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
# Set your OpenAI API key, or load from .env / environment variable
os.environ["OPENAI_API_KEY"] = os.environ.get("OPENAI_API_KEY", "your-key-here")


In [3]:
import parawave
import json
from parawave import SharedState
from openai import AsyncOpenAI
from jinja2 import Template

client = AsyncOpenAI()

# Two topics, each repeated 10 times, interleaved
topics = ["Science", "History"] * 10  # [Science, History, Science, History, ...]
data = [{"topic": t} for t in topics]
print(f"{len(data)} items: {[d['topic'] for d in data[:6]]}...")

20 items: ['Science', 'History', 'Science', 'History', 'Science', 'History']...


## Run A: Without SharedState (baseline)

Generate 20 QA pairs (10 per topic) without any feedback between generations. The LLM sees the same prompt every time for each topic.

In [4]:
@parawave(max_concurrency=2, progress="console")
async def generate_baseline(topic: str) -> dict:
    response = await client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": "Generate a unique question-answer pair. Return JSON with 'question' and 'answer' keys."},
            {"role": "user", "content": f"Generate a QA pair about {topic}."},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

result_a = generate_baseline.run(data=data)
print(result_a.summary)

[parawave] Starting run-5720a99d6b29 | 20 items | concurrency=2


[parawave] 1/20 (1 completed) | 0.9s | 1.1 items/s


[parawave] 3/20 (3 completed) | 1.8s | 1.6 items/s


[parawave] 4/20 (4 completed) | 2.4s | 1.6 items/s


[parawave] 6/20 (6 completed) | 3.2s | 1.9 items/s


...

[parawave] 17/20 (17 completed) | 9.1s | 1.9 items/s


[parawave] 19/20 (19 completed) | 9.9s | 1.9 items/s


[parawave] 20/20 (20 completed) | 9.9s | 2.0 items/s


[parawave] Completed run-5720a99d6b29 | 20/20 completed | 9.9s | 2.0 items/s


20/20 completed | 9.9s | 2.0 items/s


In [5]:
print("=== Run A: Without SharedState ===\n")
for topic in ["Science", "History"]:
    items = [item for item in result_a if item.input["topic"] == topic]
    print(f"--- {topic} ({len(items)} pairs) ---")
    for item in items:
        print(f"  Q: {item.output['question']}")
    print()

=== Run A: Without SharedState ===

--- Science (10 pairs) ---
  Q: Why do seasons change on Earth?
  Q: What is the greenhouse effect, and why does it matter for Earth's climate?
  Q: What is photosynthesis, and where in plant cells does it primarily occur?
  Q: Why does the sky look blue during the day?
  Q: What is the greenhouse effect and why does it matter for Earth’s climate?
  Q: What is the difference between weather and climate?
  Q: Why does Earth experience different seasons during the year?
  Q: What is the difference between an atom and a molecule?
  Q: What is the difference between an exothermic and an endothermic chemical reaction?
  Q: What is the difference between an atom and a molecule?

--- History (10 pairs) ---
  Q: Which event is widely considered to have marked the beginning of the Cold War between the United States and the Soviet Union?
  Q: What was the significance of the Magna Carta in 1215?
  Q: What event is commonly used to mark the beginning of the Fre

## Run B: With SharedState (diverse)

Same input, same concurrency. But now each generation sees what was previously generated for that topic. With `max_concurrency=2` and interleaved topics, by iteration 2 each topic is guaranteed to have prior results in state.

In [6]:
state = SharedState({"Science": [], "History": []})

prompt_template = Template("""Generate a unique question-answer pair about {{ topic }}.
Return JSON with "question" and "answer" keys.
{% if previous %}
IMPORTANT: The following QA pair was already generated for this topic. Make yours DIFFERENT:
- Q: {{ previous.question }}
  A: {{ previous.answer }}
{% endif %}""")

def on_done(item):
    state.append(item.input["topic"], item.output)

@parawave(
    max_concurrency=2,
    on_item_complete=[on_done],
    progress="console",
)
async def generate_diverse(topic: str) -> dict:
    # Get last generated pair for this topic (if any)
    history = state.get(topic)
    previous = history[-1] if history else None

    prompt = prompt_template.render(topic=topic, previous=previous)

    response = await client.chat.completions.create(
        model="gpt-5.4-nano",
        messages=[
            {"role": "system", "content": "Generate a unique question-answer pair. Return JSON with 'question' and 'answer' keys."},
            {"role": "user", "content": prompt},
        ],
        response_format={"type": "json_object"},
    )
    return json.loads(response.choices[0].message.content)

result_b = generate_diverse.run(data=data)
print(result_b.summary)

[parawave] Starting run-81a4dd4f4f46 | 20 items | concurrency=2


[parawave] 1/20 (1 completed) | 0.7s | 1.5 items/s


[parawave] 3/20 (3 completed) | 1.5s | 2.0 items/s


[parawave] 5/20 (5 completed) | 2.4s | 2.0 items/s


[parawave] 7/20 (7 completed) | 3.4s | 2.1 items/s


...

[parawave] 18/20 (18 completed) | 10.3s | 1.7 items/s


[parawave] 19/20 (19 completed) | 10.8s | 1.8 items/s


[parawave] 20/20 (20 completed) | 11.3s | 1.8 items/s


[parawave] Completed run-81a4dd4f4f46 | 20/20 completed | 11.3s | 1.8 items/s


20/20 completed | 11.3s | 1.8 items/s


In [7]:
print("=== Run B: With SharedState ===\n")
for topic in ["Science", "History"]:
    items = [item for item in result_b if item.input["topic"] == topic]
    print(f"--- {topic} ({len(items)} pairs) ---")
    for item in items:
        print(f"  Q: {item.output['question']}")
    print()

=== Run B: With SharedState ===

--- Science (10 pairs) ---
  Q: What is the process by which plants convert light energy into chemical energy stored in glucose?
  Q: How does the process of DNA replication ensure that genetic information is copied accurately before a cell divides?
  Q: How does the human body use ATP to power cellular processes, and what happens to ATP during that use?
  Q: Why do scientists use the term “half-life” when describing radioactive decay, and what does it tell us about how much of a radioactive sample remains over time?
  Q: In the carbon cycle, what is the difference between carbon stored in the atmosphere as CO2 and carbon stored in rocks as carbonates, and how does CO2 eventually get locked into rocks?
  Q: How does the greenhouse effect work, and why is it considered essential for life on Earth?
  Q: What is CRISPR-Cas9, and how does it allow scientists to edit specific DNA sequences in a cell?
  Q: What is the difference between weathering and erosion

## Compare

Look at the questions above. Run A tends to repeat similar questions for the same topic. Run B produces more varied questions because each generation is aware of what came before.

In [8]:
print(f"Run A: {result_a.summary}")
print(f"Run B: {result_b.summary}")
print(f"\nSharedState accumulated {len(state.get('Science'))} Science + {len(state.get('History'))} History pairs")

Run A: 20/20 completed | 9.9s | 2.0 items/s
Run B: 20/20 completed | 11.3s | 1.8 items/s

SharedState accumulated 10 Science + 10 History pairs
